In [ ]:
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

import importlib
import app.prg_cestat
import pandas as pd

importlib.reload(app.prg_cestat)

from app.prg_cestat import CESTAT
from app.utils import Helper

utils = Helper()
config_path =r"..\CESTAT\config\config_cestat.json5"
if __name__ == "__main__":
    config = utils.load_json5(config_path)
    cestat = CESTAT(config)
    # --- fetch data ---
    df = cestat.get_data()

    print(f"Fetched Rows: {len(df)}")

    # --- filtering ---
    companies = ["reliance", "tata", "adani"]   # <-- change this
    filtered_df = cestat.filter_data(
        df,
        companies=companies,
        filter_on="parties"   # <-- adjust if your column name differs
    )

    print(f"Filtered Rows: {len(filtered_df)}")

    # --- save output ---
    output_file = "cestat_orders.xlsx"

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        utils.write_df_safe(writer, filtered_df, "Filtered Data")

    print(f"\n Done. Data saved to {output_file}")


In [23]:
import os
import importlib
import app.prg_cestat
import pandas as pd

from app.jobs.job_cestat import CestatJob
from app.utils import Helper
from app.logger import setup_logger
from app.mailer import Mailer
from app.konstant import DATA_DIR

importlib.reload(app.prg_cestat)
importlib.reload(app.jobs.job_cestat)

def cestat_runner():

    utils = Helper()

    # --- config ---
    config_path = r"..\cestat\config\config_cestat.json5"
    config = utils.load_json5(config_path)

    # --- logger ---
    logger = setup_logger(name="cestat", set_global=True)


    # --- mailer ---
    mailer = Mailer()

    # --- job ---
    job = CestatJob(
        data_dir=os.path.join(DATA_DIR, "cestat"),
        config=config,
        logger=logger,
        mailer=mailer
    )

    # --- run ---
    return job.run()


if __name__ == "__main__":
    cestat_runner()

2026-04-20 22:57:59 [INFO]: Running CESTAT DATA
2026-04-20 22:57:59 [INFO]: Getting Token...


2026-04-20 22:57:59 [INFO]: Token: 8a889148c61bcec21cad016f830d119e
2026-04-20 22:58:04 [INFO]: [POST] bench=AHMEDABAD, order_type=daily/interim -> https://cestat.gov.in/ajax/order-status-web
2026-04-20 22:58:04 [WARNING]: No data for bench=124438 (city=AHMEDABAD), order_type=daily/interim
2026-04-20 22:58:09 [INFO]: [POST] bench=ALLAHABAD, order_type=daily/interim -> https://cestat.gov.in/ajax/order-status-web
2026-04-20 22:58:10 [INFO]: [OK] Extracted 1 rows so far (bench=ALLAHABAD)
2026-04-20 22:58:20 [INFO]: [POST] bench=BANGALORE, order_type=daily/interim -> https://cestat.gov.in/ajax/order-status-web
2026-04-20 22:58:20 [WARNING]: No data for bench=129525 (city=BANGALORE), order_type=daily/interim
2026-04-20 22:58:25 [INFO]: [POST] bench=CHANDIGARH, order_type=daily/interim -> https://cestat.gov.in/ajax/order-status-web
2026-04-20 22:58:25 [INFO]: [OK] Extracted 7 rows so far (bench=CHANDIGARH)
2026-04-20 22:58:35 [INFO]: [POST] bench=CHENNAI, order_type=daily/interim -> https://

c:\Users\rando\Office Projects\cestat\app\prg_cestat.py:150: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  


In [6]:
import requests
from bs4 import BeautifulSoup
import csv
import calendar
import time

session = requests.Session()

# Step 1: Get CSRF token from form page
form_url = "https://cestat.gov.in/order-status"
resp = session.get(form_url, verify=True)
soup = BeautifulSoup(resp.text, "html.parser")
csrf_token = soup.find("input", {"name": "csrf_token"})["value"]

post_url = "https://cestat.gov.in/ajax/order-status-web"

results = []

['124438', '109120', '129525', '104044', '133568', '107079', '136507', '119315', '127482']

# Step 2: Loop through months of 2025
# for month in range(1, 13):
#     from_date = f"01-{month:02d}-2025"
#     last_day = calendar.monthrange(2025, month)[1]
#     to_date = f"{last_day:02d}-{month:02d}-2025"

#     payload = {
#         "tab": 4,
#         "csrf_token": csrf_token,
#         "bench": 127482,
#         "from": from_date,
#         "to": to_date,
#         "captcha_code": "111111",   # safe to leave blank
#         "data_table_length": 200
#     }

#     response = session.post(post_url, data=payload, verify=True)
#     response_json = response.json()

#     # Step 3: Parse JSON rows
#     for row in response_json.get("data", []):
#         serial = row[0]
#         case_no = row[1]
#         parties = row[2] #row[2].replace("<br>", " vs ")
#         date = row[3]

#         # Extract href from HTML
#         soup = BeautifulSoup(row[4], "html.parser")
#         link_tag = soup.find("a")
#         pdf_url = None
#         if link_tag and link_tag.get("href"):
#             href = link_tag["href"].replace("./", "")
#             pdf_url = f"https://cestat.gov.in/{href}"

#         results.append({
#             "month": month,
#             "serial": serial,
#             "case_no": case_no,
#             "parties": parties,
#             "date": date,
#             "pdf_url": pdf_url
#         })
    
#     time.sleep(5)   

from_date = f"01-02-2025"
# last_day = calendar.monthrange(2025, month)[1]
to_date = f"12-02-2025"

payload = {
    "tab": 4,
    "csrf_token": csrf_token,
    "bench": 127482,
    "from": from_date,
    "to": to_date,
    "captcha_code": "111111",   # safe to leave blank
    "data_table_length": 200
}

response = session.post(post_url, data=payload, verify=True)
response_json = response.json()

# Step 3: Parse JSON rows
for row in response_json.get("data", []):
    serial = row[0]
    case_no = row[1]
    parties = row[2].replace("<br>", "") #row[2].replace("<br>", " vs ")
    date = row[3]
    
    p1,p2 = parties.split("vs")

    # Extract href from HTML
    soup = BeautifulSoup(row[4], "html.parser")
    link_tag = soup.find("a")
    pdf_url = None
    if link_tag and link_tag.get("href"):
        href = link_tag["href"].replace("./", "")
        pdf_url = f"https://cestat.gov.in/{href}"

    results.append({
        # "month": month,
        "serial": serial,
        "case_no": case_no,
        "parties": parties,
        "p1":p1.lower().strip(),
        "p2":p2.lower().strip(),
        "date": date,
        "pdf_url": pdf_url
    })

time.sleep(5)   


# Step 4: Save results to CSV
with open(f"cestat_{payload["bench"]}_2025.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["serial", "case_no","parties" ,"p1","p2", "date", "pdf_url"])
    writer.writeheader()
    writer.writerows(results)

# print("Saved results to cestat_cases_4_2025.csv")